# Tuning, final model and deployment check (phase 2)

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100) · Settings → **Internet ON** · *Add Input* → your dataset (the folder with `manifest.csv`).

The whole protocol lives in `src/bus/tune.py`; each cell below runs one stage. Every stage skips work that is already finished, so after a session ends you can attach this notebook's output as input, run the **Resume** cell, and run the remaining cells again.

| Section | What it does | Rough GPU time |
|---|---|---|
| 1 | unit tests + the whole protocol on a few images (code check) | 10 min |
| 2 | plain U-Net with the fix (paper Table 1 row) | 15 min |
| 3 | classifier grid: 3 backbones × 4 crop contexts × 2 fine-tuning depths × 3 seeds = 72 runs | 1.5–2.5 h |
| 4 | segmentation grid: 2 architectures × with/without photometric augmentation × 3 seeds = 12 runs | 1 h |
| 5 | retrain the best models on BUSI + Breast-Lesions-USG + BUS-UCLM (3 seeds) | 1 h |
| 6 | choose the final system on validation data; pick the decision threshold | 15 min |
| 7 | test baseline (= reproduction) vs final on BUSI, all test sets and BUS-BRA | 20 min |
| 8 | export the deployment bundle, try it on an image, test the web API | 5 min |
| 9 | report and downloads | 2 min |

For the long run use **Save Version → Save & Run All (Commit)**: it keeps running when you close the browser.

## 0. Setup

In [ ]:
import os, sys, importlib
REPO_URL = "https://github.com/vp-y/mtp.git"
REPO = "/kaggle/working/repo"

# Only if the repo is PRIVATE: add a Kaggle secret GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

!rm -rf {REPO}
!git clone --depth 1 -b main {REPO_URL} {REPO}
assert os.path.isdir(f"{REPO}/src/bus"), "Clone FAILED - read the git message above"
%cd {REPO}
!pip install -q -r requirements-kaggle.txt

# make the 'bus' package visible to this notebook AND to every !python command
SRC = f"{REPO}/src"
os.environ["PYTHONPATH"] = SRC
sys.path_importer_cache.pop(SRC, None)
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
!python -c "import bus; print('bus OK ->', bus.__file__)"

In [ ]:
import glob
hits = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
assert hits, "Dataset not found: add the dataset that contains manifest.csv via 'Add Input'"
DATA = os.path.dirname(hits[0])
os.environ["BUS_DATA_ROOT"] = DATA
print("data:", DATA)
!nvidia-smi --query-gpu=name,memory.total --format=csv

T = "/kaggle/working/tuning"        # all tuning results go here

In [ ]:
# Resume (only in a NEW session): attach the previous version's output as input, then uncomment:
# !cp -r /kaggle/input/YOUR-PREVIOUS-OUTPUT/tuning /kaggle/working/

## 1. Check the code
Unit tests, then every tuning stage on 8 images for 1 epoch (2 configurations per grid, 2 seeds). If anything here fails, stop and send the error.

In [ ]:
!python -m pytest -q tests
!python -m bus.tune all --out /kaggle/working/tune_smoke --smoke
!rm -rf /kaggle/working/tune_smoke

## 2. Plain U-Net with the fix
He initialisation (as in the original U-Net paper), at least 30 epochs, learning-rate schedule on validation loss. Also a BatchNorm variant for comparison.

In [ ]:
!python -m bus.tune vanilla --out {T}

## 3. Classifier grid (Scenario A, ground-truth crops, 3 seeds)
Crop context: `m0.1` paper-style crop · `m0.5` box ×2 · `m1.0` box ×3 · `full` whole image. Fine-tuning: `k10` last 10 tensors (paper) · `kall` whole backbone at a 10× lower learning rate.

In [ ]:
!python -m bus.tune cls --out {T}

## 4. Segmentation grid (Scenario A, 3 seeds)
U-Net vs DeepLabV3+ (ResNet34, 64 channels), each with and without photometric augmentation (brightness, contrast, gamma, speckle, blur).

In [ ]:
!python -m bus.tune seg --out {T}

## 5. Retrain the best configurations on more data
BUSI + Breast-Lesions-USG + BUS-UCLM (scenario `deploy`), 3 seeds; classifier with and without photometric augmentation.

In [ ]:
!python -m bus.tune deploy --out {T}

## 6. Choose the final system (validation data only)
Candidates: baseline (reproduction), best single models, 3-seed ensembles trained on BUSI or on all three datasets, ± flip test-time augmentation. Chosen by validation AUC on the common validation set; then mask post-processing and the decision threshold for 90 % validation sensitivity.

In [ ]:
!python -m bus.tune select --out {T}

## 7. Final test: baseline vs final system
Done once. Test sets: BUSI + Breast-Lesions-USG + BUS-UCLM test (with normal scans), BUSI test (as in the paper), BUS-BRA (external).

In [ ]:
!python -m bus.tune final --out {T}

## 8. Deployment bundle
Export the final models + settings, run the bundle on one external image, and send one request to the web API (FastAPI test client).

In [ ]:
!python -m bus.tune export --out {T}
!python -m bus.bundle info {T}/deploy_bundle

In [ ]:
import cv2, matplotlib.pyplot as plt, pandas as pd
from bus.bundle import load_bundle
pipe, settings = load_bundle(f"{T}/deploy_bundle")
m = pd.read_csv(f"{DATA}/manifest.csv")
row = m[(m.dataset == "BUS-BRA") & (m.label == "malignant")].iloc[0]
gray = cv2.imread(f"{DATA}/{row.image_path}", cv2.IMREAD_GRAYSCALE)
o = pipe(gray)
print(f"{row.id} (truth: {row.label}) -> lesion found: {o['lesion_found']}, "
      f"P(malignant) = {o['prob_malignant']:.3f}, threshold = {settings['threshold']:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(gray, cmap="gray"); ax[0].set_title("image")
ax[1].imshow(gray, cmap="gray"); ax[1].imshow(o["mask"], alpha=0.35, cmap="autumn"); ax[1].set_title("predicted lesion")
for a in ax: a.axis("off")
plt.show()

In [ ]:
# Web API check: start the app in-process and send one image, exactly as the browser UI does
!pip install -q "fastapi>=0.100" python-multipart httpx
os.environ["BUS_BUNDLE"] = f"{T}/deploy_bundle"
sys.path.insert(0, REPO)
from fastapi.testclient import TestClient
from app.server import app
with TestClient(app) as client:
    print(client.get("/api/health").json())
    ok, png = cv2.imencode(".png", gray)
    r = client.post("/api/predict", files={"file": ("test.png", png.tobytes(), "image/png")})
    d = r.json()
    print(r.status_code, {k: d[k] for k in ("decision", "prob_malignant", "threshold", "lesion_area_pct", "elapsed_ms")})
    assert r.status_code == 200 and client.get("/").status_code == 200
print("web app OK")

## 9. Report and downloads

In [ ]:
!python -m bus.tune report --out {T}
from IPython.display import Markdown, display
display(Markdown(open(f"{T}/reports/tuning_report.md").read()))

In [ ]:
%cd /kaggle/working
!rm -f tuning_results.zip deploy_bundle.zip
!zip -qr tuning_results.zip tuning -x "*.pt" "*/pred_masks/*" "tuning/deploy_bundle/*"
!cd tuning && zip -qr ../deploy_bundle.zip deploy_bundle
!ls -lh tuning_results.zip deploy_bundle.zip
from IPython.display import FileLink
display(FileLink("tuning_results.zip")); display(FileLink("deploy_bundle.zip"))